In [53]:
!pip install pyserial

In [57]:
!pip uninstall pandas -y
!pip install pandas

Found existing installation: pandas 2.2.3
Uninstalling pandas-2.2.3:
  Successfully uninstalled pandas-2.2.3
  Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl (11.5 MB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.32.0 requires numpy<2,>=1.19.3, but you have numpy 2.2.2 which is incompatible.


In [31]:
import cv2
import os
import time
import numpy as np
import serial
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Layer, Dropout

# Define the custom FixedDropout layer
class FixedDropout(Layer):
    def __init__(self, rate, **kwargs):
        super(FixedDropout, self).__init__(**kwargs)
        self.rate = rate
    def call(self, inputs, training=None):
        return tf.nn.dropout(inputs, rate=self.rate)

# Load the VGG16 model from the .h5 file
model = load_model(r'C:/Users/apple/OneDrive/Desktop/models/model_VGG.h5',
custom_objects={'FixedDropout': Dropout})

class_labels = ['unripe', 'preripe', 'ripe', 'rotten']

#Initialize webcam capture
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

# Directory to save captured images
save_dir = "captured_images"
os.makedirs(save_dir, exist_ok=True)

image_count = 0
saved_image_paths = []

def write_read(x):
    """Send data to Arduino and read response."""
    arduino.write(bytes(x, 'utf-8'))
    time.sleep(0.05)
    return arduino.readline()

while True:
    ret, frame = cap.read()
    
    if not ret:
        print("Error: Failed to capture frame.")
        break

    # Save the image when space bar is pressed
     key = cv2.waitKey(1)
     if key == ord(' '): # Space bar
        image_count += 1
        image_path = os.path.join(save_dir, f'image_{image_count}.jpg')
        cv2.imwrite(image_path, frame)
        saved_image_paths.append(image_path)
        print(f"Image saved: {image_path}")

    # Preprocess the saved image for model input
    resized_frame = cv2.resize(frame, (224, 224))  # Assuming VGG16 input size
    preprocessed_frame = preprocess_input(resized_frame)
    input_image = np.expand_dims(preprocessed_frame, axis=0)

    # Get model predictions (assuming `model` and `class_labels` are defined)
    predictions = model.predict(input_image)
    max_prob_index = np.argmax(predictions)
    class_label = class_labels[max_prob_index]
    probability = predictions[0][max_prob_index]

# Print the predicted class and its probability
    print(f"Predicted Class: {class_label}, Probability: {probability:.2f}")


AttributeError: partially initialized module 'pandas' has no attribute '_pandas_parser_CAPI' (most likely due to a circular import)

In [199]:
#Send the predicted class label to Arduino
#label = class_label.encode()
#x = type(label)
#print(x)
#label_string = label.decode()
#print(label_string)
# x = type(label_string)
#print(x)
#value = write_read(label_string)

In [ ]:
# Break the loop if 'q' is pressed 
    elif key & 0xFF == ord('q'):
    break

# Release the capture
cap.release()
cv2.destroyAllWindows()

# Cleanup: Delete the saved images
for image_path in saved_image_paths:
    os.remove(image_path)
print("Saved images deleted.")